In [1]:
import pandas as pd
import numpy as np
from src.kmnri import kmnri
from src.process_nri import process_nri
from src.categ_nri_boot import categ_nri_boot

In [2]:
data = pd.read_csv("./data/data.csv")
data = data.loc[:, ["status", "time", "pred_base", "pred_big"]]
data.insert(4, "pred_big2", data.pred_big * 1.1)

In [3]:
preds = ["pred_base", "pred_big", "pred_big2"]
event_time = "time"
event = "status"
time_span = 600
r_boot_nri = 50
risklimits_nri_score = [0.9]

In [4]:
def nri_surv(preds, data, risklimits, ftime, censvar, Time, Nboot):

    def wrapper (pred):
        result = categ_nri_boot(risklimits=risklimits,
                                pold=data[f"{preds[0]}"],
                                pnew=data[f"{pred}"],
                                ftime=data[ftime],
                                censvar=data[censvar],
                                Time=Time,
                                Nboot=Nboot)
        return result

    nri_matrix_ac = [wrapper(pred) for pred in preds[1:]]
    nri_matrix_ac = pd.DataFrame(nri_matrix_ac)
    nri_matrix_ac = process_nri(nri_matrix_ac)
    
    return nri_matrix_ac

In [5]:
nri_surv(
    preds=preds,
    data=data,
    risklimits=risklimits_nri_score,
    ftime=event_time,
    censvar=event,
    Time=time_span,
    Nboot=r_boot_nri,
)

/Users/janbrederecke/Desktop/nri/src/categ_nri_boot.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  censvar[v] = 0
/Users/janbrederecke/Desktop/nri/src/categ_nri_boot.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ftime[v] = Time


,nri,nri.left,nri.right,nri.ev,nri.ev.left,nri.ev.right,nri.nev,nri.nev.left,nri.nev.right,N,Nevent
0,-0.042843,-0.093386,0.007699,-0.022309,-0.049409,0.004792,-0.020535,-0.067478,0.026408,228,148
1,0.126712,0.026556,0.226867,0.198117,0.148445,0.247788,-0.071405,-0.152279,0.009469,228,148


In [6]:
# def wrapper(pred):
#     result = categ_nri_boot(risklimits=risklimits_nri_score,
#                             pold=data[f"{preds[0]}"],
#                             pnew=data[f"{pred}"],
#                             ftime=data[event_time],
#                             censvar=data[event],
#                             Time=time_span,
#                             Nboot=r_boot_nri)
#     return result

In [7]:
# nri_matrix_ac = [wrapper(pred) for pred in preds[1:]]
# nri_matrix_ac = pd.DataFrame(nri_matrix_ac)
# nri_matrix_ac = process_nri(nri_matrix_ac)